# 0. Imoprt packets

In [ ]:
from Adversarial_training import get_adv_data,train_loop,test_loop
from ATKMethods import generatePerturbData,AlexNetTF
from sklearn.model_selection import train_test_split
import numpy as np
import utils.gestureDataLoader as gestureDataLoader
import tensorflow as tf
import os
import utils.Config as Config
import random
random.seed(42)
config = Config.getconfig( )

# 1. Load Dataset

Load SignFi Dataset

In [2]:
config.source = 'lab_276'
assert config.source != None, 'source should not be None'
train_data, test_data, train_label, test_label = gestureDataLoader.getData(
        config, 'signfi'
        )

X_train, X_test, y_train, y_test = train_test_split( train_data, train_label, test_size=0.1, random_state=42)
batch_size = config.batch_size
# Prepare the training dataset.
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size)

val_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
val_dataset = val_dataset.batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices((test_data, test_label))
test_dataset = test_dataset.batch(batch_size)

# 2. Configuration

In [3]:
config.DNN_name = 'defult'
psr = 0.001
method = 'pgd'
n_iter = 3
model_name = f'robust_adv_training_{method}_psr_{psr}_{config.DNN_name}_{config.source}_niter_{n_iter}.h5'
config.model_path['adv_robust_model_path'] = os.path.join( config.model_path['adv_robust_model_path'], model_name)
net = AlexNetTF( config )
model = net.buildModel( choice = config.DNN_name)

# 3. Adversarial Training

In [ ]:
model = train_loop(config,model,train_dataset,val_dataset,psr,method,n_iter = n_iter)

# Evaluation

In [14]:
acc_all_adv = {}
eva_model_list = {
    'Normal':'SavedModel\\PSR\\signfi_model_lab_276_scale_1.h5',
    'fgsm':'SavedModel\\Adversarial_robust_model\\robust_adv_training_fgsm_psr_0.001_defult_lab_276_niter_None.h5',
    'PGD_3':'SavedModel\\Adversarial_robust_model\\robust_adv_training_pgd_psr_0.001_defult_lab_276_niter_3.h5',
}

attacker_method = 'pgd'
for iter in [2,3,4]:
    for adv_training_method, model_path in eva_model_list.items():
        acc_buf = []
        model.load_weights(model_path)
        print(f'==========================================Adversarial training by {adv_training_method}========================================================')
        for psr_current in [0.0, 6.25e-05, 0.000125, 0.0001875, 0.00025, 0.0003125, 0.000375, 0.0004375, 0.0005]:
            
            test_acc = test_loop(config,psr_current,model,test_dataset,attacker_method,n_iter = iter)
            
            acc_buf.append( test_acc )
            print(f'Adversarial attack method: {adv_training_method}, psr_current: {psr_current:.6f}, accuracy: {test_acc:.6f}')
        acc_all_adv.update( {adv_training_method +'_defense_against_'+ attacker_method + f'_{iter}': np.asarray(acc_buf)} )
    



==========================================Adversarial training by Normal========================================================
Adversarial attack method: Normal, psr_current: 0.000000, accuracy: 0.909420
Adversarial attack method: Normal, psr_current: 0.000063, accuracy: 0.835145
Adversarial attack method: Normal, psr_current: 0.000125, accuracy: 0.781703
Adversarial attack method: Normal, psr_current: 0.000188, accuracy: 0.728261
Adversarial attack method: Normal, psr_current: 0.000250, accuracy: 0.684783
Adversarial attack method: Normal, psr_current: 0.000313, accuracy: 0.624094
Adversarial attack method: Normal, psr_current: 0.000375, accuracy: 0.587862
Adversarial attack method: Normal, psr_current: 0.000438, accuracy: 0.550725
Adversarial attack method: Normal, psr_current: 0.000500, accuracy: 0.508152
==========================================Adversarial training by fgsm========================================================
Adversarial attack method: fgsm, psr_current: 0.000

In [15]:
print(acc_all_adv)

{'Normal_defense_against_pgd_2': array([0.9094203 , 0.83514494, 0.7817029 , 0.7282609 , 0.6847826 ,
       0.6240942 , 0.5878623 , 0.5507246 , 0.5081522 ], dtype=float32), 'fgsm_defense_against_pgd_2': array([0.7246377 , 0.72282606, 0.7201087 , 0.7173913 , 0.7137681 ,
       0.7119565 , 0.7119565 , 0.7092391 , 0.70742756], dtype=float32), 'PGD_3_defense_against_pgd_2': array([0.8007246 , 0.7961956 , 0.7952899 , 0.78985506, 0.78532606,
       0.7826087 , 0.7780797 , 0.7753623 , 0.76992756], dtype=float32), 'Normal_defense_against_pgd_3': array([0.9094203 , 0.8387681 , 0.79710144, 0.75      , 0.7047101 ,
       0.6494565 , 0.61503625, 0.58967394, 0.55163044], dtype=float32), 'fgsm_defense_against_pgd_3': array([0.7246377 , 0.72282606, 0.7201087 , 0.7182971 , 0.7155797 ,
       0.7128623 , 0.7119565 , 0.71105075, 0.7092391 ], dtype=float32), 'PGD_3_defense_against_pgd_3': array([0.8007246, 0.7961956, 0.7952899, 0.7916667, 0.7871377, 0.7835145,
       0.7807971, 0.7762681, 0.7744565], dtyp

In [ ]:
print(f'adversarial model on clean example {test_loop(config,0.0,model,test_dataset,None)*100:.4f} %',)

In [ ]:

# from scipy.io import loadmat
def comp_atk_success_rate(acc_array):
    return (acc_array[0] - acc_array)/acc_array[0]

for method, acc in acc_all_adv.items():
    print( f'The attack success rate with adversarial training {method}', comp_atk_success_rate(acc) )

